<a href="https://colab.research.google.com/github/Andrian0s/ML4NLP1-2025-Tutorial-Notebooks/blob/main/tutorials_notebooks_in_class_2025/W09_introduction_to_gliner_nuextract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Introduction to GLiNER**

## Model information

GLiNER (Generalist Lightweight Named Entity Recognizer) is a **modern, efficient model for Named Entity Recognition (NER)**.
Unlike traditional NER systems that require task-specific fine-tuning, GLiNER can **generalize across domains and entity types** with **minimal supervision**.

It’s particularly useful when you need:

* **Zero-shot NER**, without retraining large models.
* **Fast inference** with limited computational resources.
* **Custom entity extraction** from text (e.g., companies, genes, product names, etc.).

### How GLiNER Works

GLiNER reframes NER as a **text-text matching problem**:

* Entities and text are placed in the same prompt.
* Span embeddings are calculated by using up to 12 consecutive tokens: First and last token represantations are concatenated and passed through a feed-forward network for refinement.
* Entity embeddings are calculated by taking the **first** token of each entity's token representations and passing it through a feed-forward network for refinement (different than the span's FFN).
* Entity and span embeddings are in the same latent space.
* Scores are calculated between all possible combinations of (entity embedding) <-> (span embedding)


More details in this [paper.](https://arxiv.org/abs/2311.08526)

## Trying out GLiNER

You can experiment with the model on this [HuggingFace space](https://huggingface.co/spaces/urchade/gliner_multiv2.1).

In [ ]:
from gliner import GLiNER


model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")
# NOTE: You can load it into your GPU with .to("cuda")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

gliner_config.json:   0%|          | 0.00/477 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.16G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [ ]:
text = "Marie Curie discovered radium in Paris in 1898."

# Define entity types in natural language
entity_descriptions = [
  "person name",
  "chemical element",
  "city",
  "date",
  "id number"
]

# Run prediction
entities = model.predict_entities(
    text,
    entity_descriptions,
)
entities

[{'start': 0,
  'end': 11,
  'text': 'Marie Curie',
  'label': 'person name',
  'score': 0.9769240021705627},
 {'start': 23,
  'end': 29,
  'text': 'radium',
  'label': 'chemical element',
  'score': 0.9522218704223633},
 {'start': 33,
  'end': 38,
  'text': 'Paris',
  'label': 'city',
  'score': 0.9851381778717041},
 {'start': 42,
  'end': 46,
  'text': '1898',
  'label': 'date',
  'score': 0.9774957895278931}]

You can run inference in parallalel with .run and by setting the batch_size argument! Load the model in GPU to see performace gains.

In [ ]:
texts = ["Marie Curie discovered radium in Paris in 1898."] * 10

# Define entity types in natural language
entity_descriptions = [
  "person name",
  "chemical element",
  "city",
  "date",
  "id number"
]

# Run prediction
entities = model.run(
    texts,
    entity_descriptions,
    batch_size=5
)
entities

[[{'start': 0,
   'end': 11,
   'text': 'Marie Curie',
   'label': 'person name',
   'score': 0.9769238829612732},
  {'start': 23,
   'end': 29,
   'text': 'radium',
   'label': 'chemical element',
   'score': 0.9522213339805603},
  {'start': 33,
   'end': 38,
   'text': 'Paris',
   'label': 'city',
   'score': 0.9851382970809937},
  {'start': 42,
   'end': 46,
   'text': '1898',
   'label': 'date',
   'score': 0.977495551109314}],
 [{'start': 0,
   'end': 11,
   'text': 'Marie Curie',
   'label': 'person name',
   'score': 0.9769238829612732},
  {'start': 23,
   'end': 29,
   'text': 'radium',
   'label': 'chemical element',
   'score': 0.9522213339805603},
  {'start': 33,
   'end': 38,
   'text': 'Paris',
   'label': 'city',
   'score': 0.9851382970809937},
  {'start': 42,
   'end': 46,
   'text': '1898',
   'label': 'date',
   'score': 0.977495551109314}],
 [{'start': 0,
   'end': 11,
   'text': 'Marie Curie',
   'label': 'person name',
   'score': 0.9769238829612732},
  {'start': 2

# **Introduction to NuExtract (NOTE: This is not included in the exercise)**

## Model information

**NuExtract (v1.5)** is a multilingual, long-context model for structured information extraction.
Instead of focusing on fixed entity types (like traditional NER), it receives s JSON template and a text, and the model responds with the same JSON, but with values **extracted** from the text.

It’s particularly useful when you need:

* **Schema-guided extraction**, not just label classification.
* **Long document support** (up to ~20 K tokens).
* **Multilingual extraction** across English, French, German, Spanish, and more.
* **Extractive outputs**, where values come directly from the input text.

### How NuExtract Works

NuExtract reframes extraction as a **structured generation task** guided by a JSON template:

* Inputs combine a **text document** and a **JSON schema** describing the fields to extract (few shot examples for the specific JSON schema can also be used).
* The model (a fine-tuned **Phi-3.5-mini-instruct** LLM) generates a filled-in JSON, extracting values verbatim from the text.
* For long texts, it uses a **sliding-window** strategy — chunking the text, extracting per chunk, then merging results.
* Training aligns the model to fill schemas accurately across languages and domains.

More details on the model [here.](https://huggingface.co/numind/NuExtract-1.5)


## Trying out NuExtract

You can experiment with the model on this [HuggingFace space](https://huggingface.co/numind/NuExtract-1.5). Copy paste text from an article of your choice, define your custom JSON template and run inference.

### **Example:**

****Text:****

Natural language processing has its roots in the 1950s. Already in 1950, Alan Turing published an article titled "Computing Machinery and Intelligence" which proposed what is now called the Turing test as a criterion of intelligence, though at the time that was not articulated as a problem separate from artificial intelligence. The proposed test includes a task that involves the automated interpretation and generation of natural language.

****JSON template:****
```json
{
  "topic": "",
  "time_period": "",
  "key_figures": [
    {
      "name": "",
      "contribution": ""
    }
  ],
  "important_publications": [
    {
      "title": "",
      "year": "",
      "author": ""
    }
  ],
  "concepts_introduced": [
    {
      "name": "",
      "description": ""
    }
  ],
  "relationship_to_other_fields": ""
}
```



****Model's output:****
```json
{
    "topic": "Natural language processing",
    "time_period": "1950s",
    "key_figures": [
        {
            "name": "Alan Turing",
            "contribution": "published an article titled \"Computing Machinery and Intelligence\" which proposed what is now called the Turing test as a criterion of intelligence"
        }
    ],
    "important_publications": [
        {
            "title": "Computing Machinery and Intelligence",
            "year": "1950",
            "author": "Alan Turing"
        }
    ],
    "concepts_introduced": [
        {
            "name": "Turing test",
            "description": "a criterion of intelligence"
        }
    ],
    "relationship_to_other_fields": ""
}
```